In [1]:
import os
		
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

d:\miniconda3\envs\PyTorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [4]:
ds = ds.train_test_split(test_size=0.2)
ds

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 21486
    })
    test: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 5372
    })
})

In [5]:
ds['train'][:3]

{'output': ['该查询可用SQL（结构化查询语言）编写，如下：\n\n```\nSELECT 姓名 FROM 员工\nWHERE 经验 > 10;\n```',
  '一项可以在室内完成的手工活动是制作贺毡动物。这是一个非常有趣和令人放松的活动，可以帮助提高创造力和手工技能。您所需要的材料是：彩色羊毛毡，毛线针，海绵垫或毡垫，和剪刀。然后，您可以通过不断地用毛线针将羊毛毡戳入垫子上，塑造出各种各样的动物形状。这项活动不仅可以为您带来乐趣，同时还可以让您在完成后拥有一个可爱的手工制品。',
  '一本书就像一把雨伞。它们都能在寂寞的时光为我们遮风挡雨。当我们站在雨中，被世界淋湿时，一把雨伞可以帮助我们遮挡住风雨。同样的，当生活中的压力和不确定性包围着我们时，一本好书也可以为我们提供庇护所，在阅读中让我们获得内心的平静与慰藉。雨伞可以在我们需要时为我们带来温暖，书也能在我们需要的时候提供智慧与启发，陪伴我们走过人生的风雨。'],
 'input': ['', '', ''],
 'instruction': ['从表中提取所需信息的查询：从“员工”表中检索具有超过10年经验的员工的姓名。',
  '建议一项可以在室内完成的手工活动。',
  '构建书和雨伞之间的比喻。']}

In [6]:
tokenizer = AutoTokenizer.from_pretrained("Langboat/bloom-1b4-zh")
tokenizer

BloomTokenizerFast(name_or_path='Langboat/bloom-1b4-zh', vocab_size=46145, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [7]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [8]:
tokenized_ds = ds.map(process_func, remove_columns=ds['train'].column_names)
tokenized_ds

Map: 100%|██████████| 5372/5372 [00:01<00:00, 4754.36 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21486
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5372
    })
})

In [9]:
tokenizer.decode(tokenized_ds['train'][1]["input_ids"])

'Human: 建议一项可以在室内完成的手工活动。\n\nAssistant: 一项可以在室内完成的手工活动是制作贺毡动物。这是一个非常有趣和令人放松的活动，可以帮助提高创造力和手工技能。您所需要的材料是：彩色羊毛毡，毛线针，海绵垫或毡垫，和剪刀。然后，您可以通过不断地用毛线针将羊毛毡戳入垫子上，塑造出各种各样的动物形状。这项活动不仅可以为您带来乐趣，同时还可以让您在完成后拥有一个可爱的手工制品。</s>'

In [10]:
tokenizer.decode(list(filter(lambda x : x != -100, tokenized_ds['train'][1]['labels'])))

'一项可以在室内完成的手工活动是制作贺毡动物。这是一个非常有趣和令人放松的活动，可以帮助提高创造力和手工技能。您所需要的材料是：彩色羊毛毡，毛线针，海绵垫或毡垫，和剪刀。然后，您可以通过不断地用毛线针将羊毛毡戳入垫子上，塑造出各种各样的动物形状。这项活动不仅可以为您带来乐趣，同时还可以让您在完成后拥有一个可爱的手工制品。</s>'

In [11]:
model = AutoModelForCausalLM.from_pretrained("Langboat/bloom-1b4-zh")

In [12]:
from peft import IA3Config, TaskType, get_peft_model

config = IA3Config(task_type=TaskType.CAUSAL_LM)
config

IA3Config(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.IA3: 'IA3'>, auto_mapping=None, peft_version='0.18.0', base_model_name_or_path=None, revision=None, inference_mode=False, target_modules=None, exclude_modules=None, feedforward_modules=None, fan_in_fan_out=False, modules_to_save=None, init_ia3_weights=True)

In [13]:
model = get_peft_model(model, config)
model

PeftModelForCausalLM(
  (base_model): IA3Model(
    (model): BloomForCausalLM(
      (transformer): BloomModel(
        (word_embeddings): Embedding(46145, 2048)
        (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (h): ModuleList(
          (0-23): 24 x BloomBlock(
            (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
            (self_attention): BloomAttention(
              (query_key_value): Linear(
                (base_layer): Linear(in_features=2048, out_features=6144, bias=True)
                (ia3_l): ParameterDict(  (default): Parameter containing: [torch.FloatTensor of size 6144x1])
              )
              (dense): Linear(in_features=2048, out_features=2048, bias=True)
              (attention_dropout): Dropout(p=0.0, inplace=False)
            )
            (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
            (mlp): BloomMLP(
              (dense_

In [14]:
model.print_trainable_parameters()

trainable params: 344,064 || all params: 1,303,455,744 || trainable%: 0.0264


In [15]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    logging_steps=50,
    eval_strategy='steps',
    num_train_epochs=1,
    learning_rate=3e-3
)

In [17]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds['train'].select(range(10000)),
    eval_dataset=tokenized_ds['test'].select(range(2000)),
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

C:\Users\10433\AppData\Local\Temp\ipykernel_12340\3768420791.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss


In [ ]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)